In [1]:
import torch
import torch.nn.functional as F

import random

# Data preparation

In [2]:
names = []
with open("./data/names.txt", "r") as f:
    for line in f:
        names.append(line.rstrip())

len(names)

32033

In [3]:
import random

random.seed(22334455)
random.shuffle(names)

In [4]:
names[:10]

['maitreya',
 'nakari',
 'kalessi',
 'elex',
 'jacinda',
 'skylor',
 'adalae',
 'kenn',
 'ephram',
 'capri']

In [5]:
alphabet = '.' + "".join(sorted(set("".join(names))))
alphabet

'.abcdefghijklmnopqrstuvwxyz'

In [6]:
alphabet_len = len(alphabet)

In [7]:
char_to_idx = {}
for idx, char in enumerate(alphabet):
    char_to_idx[char] = idx

char_to_idx

{'.': 0,
 'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26}

In [8]:
def create_data_sample(word, prefix_len):
    X = []
    Y = []
    
    prefix = [char_to_idx['.']] * prefix_len
    for char in word + '.':
        X.append(prefix)
        idx = char_to_idx[char]
        Y.append(idx)

        prefix = prefix[1:] + [idx]

    return X, Y

In [9]:
def index_to_word(indexes):
    return "".join(alphabet[idx] for idx in indexes)

In [10]:
def print_data(data):
    X, Y = data

    for x, y in zip(X, Y):
        prefix = index_to_word(x)

        print(f"{prefix} --> {alphabet[y]}")
        
data = create_data_sample("rahul", 3)
print_data(data)

... --> r
..r --> a
.ra --> h
rah --> u
ahu --> l
hul --> .


In [11]:
num_prefix_chars = 5

def create_dataset(words, prefix_len):
    X = []
    Y = []

    for word in words:
        x, y = create_data_sample(word, prefix_len)

        X += x
        Y += y

    return X, Y

In [12]:
def print_dataset(data, labels):
    for d, l in zip(data, labels):
        print(f"{index_to_word(d)} --> {alphabet[l]}")

In [13]:
print_dataset(*create_dataset(names[:3], 3))

... --> m
..m --> a
.ma --> i
mai --> t
ait --> r
itr --> e
tre --> y
rey --> a
eya --> .
... --> n
..n --> a
.na --> k
nak --> a
aka --> r
kar --> i
ari --> .
... --> k
..k --> a
.ka --> l
kal --> e
ale --> s
les --> s
ess --> i
ssi --> .


In [14]:
training_set_size = int(0.8 * len(names))
dev_set_size = int(0.1 * len(names))

training_set = names[:training_set_size]
dev_set = names[training_set_size:training_set_size + dev_set_size]
test_set = names[training_set_size + dev_set_size:]

print(len(training_set), len(dev_set), len(test_set))

25626 3203 3204


In [15]:
train_data, train_labels = create_dataset(training_set, num_prefix_chars)
dev_data, dev_labels = create_dataset(dev_set, num_prefix_chars)
test_data, test_labels = create_dataset(test_set, num_prefix_chars)

In [16]:
print("\n--- TRAIN ---")
print_dataset(train_data[:10], train_labels[:10])
print("\n--- DEV ---")
print_dataset(dev_data[:10], dev_labels[:10])
print("\n--- TEST ---")
print_dataset(test_data[:10], test_labels[:10])


--- TRAIN ---
..... --> m
....m --> a
...ma --> i
..mai --> t
.mait --> r
maitr --> e
aitre --> y
itrey --> a
treya --> .
..... --> n

--- DEV ---
..... --> z
....z --> v
...zv --> i
..zvi --> .
..... --> m
....m --> a
...ma --> n
..man --> u
.manu --> e
manue --> l

--- TEST ---
..... --> e
....e --> m
...em --> b
..emb --> y
.emby --> r
embyr --> .
..... --> c
....c --> r
...cr --> a
..cra --> w


In [17]:
assert len(train_data) == len(train_labels)
assert len(dev_data) == len(dev_labels)
assert len(test_data) == len(test_labels)

In [18]:
train_data = torch.tensor(train_data)
train_labels = torch.tensor(train_labels)

dev_data = torch.tensor(dev_data)
dev_labels = torch.tensor(dev_labels)

test_data = torch.tensor(test_data)
test_labels = torch.tensor(test_labels)

# Neural Network

## Embedding layer

In [19]:
class Embedding:
    def __init__(self, in_len, num_dims):
        self.in_len = in_len
        self.num_dims = num_dims

        self.embedding = torch.randn((in_len, num_dims), requires_grad=True)

    def __call__(self, index):
        return self.embedding[index]

    def parameters(self):
        return [self.embedding]

    def __repr__(self):
        return f"{self.__class__.__name__}: shape({self.embedding.shape})"

    def save_weights(self):
        return [p.detach().clone() for p in self.parameters()]

    def load_weights(self, weights):
        self.embedding = weights[0].clone().requires_grad_()

In [20]:
test_emb = Embedding(27, 2)
test_emb.embedding

tensor([[-0.7809,  0.3950],
        [ 0.3863,  1.9058],
        [ 0.8515, -0.7853],
        [-0.9000,  1.0759],
        [-1.1201, -0.2684],
        [ 1.1450,  0.1559],
        [ 1.3897, -0.5255],
        [ 0.0043,  0.2450],
        [ 0.0787, -0.5419],
        [ 0.5168, -0.5784],
        [ 1.0119,  1.9327],
        [-0.1872, -0.7539],
        [ 0.1311,  1.5708],
        [ 1.2899,  1.2745],
        [ 1.3265, -0.5562],
        [ 0.1075,  1.2083],
        [ 1.0381, -0.6970],
        [-0.1505, -0.4198],
        [-0.3288, -0.6862],
        [-0.8761,  0.3051],
        [ 0.1742, -0.0465],
        [ 0.7738,  0.5199],
        [-0.4224,  1.6996],
        [ 0.8065, -1.5457],
        [ 0.4880, -0.5391],
        [ 0.5614, -1.0774],
        [-0.9671,  0.5795]], requires_grad=True)

In [21]:
test_emb.parameters()

[tensor([[-0.7809,  0.3950],
         [ 0.3863,  1.9058],
         [ 0.8515, -0.7853],
         [-0.9000,  1.0759],
         [-1.1201, -0.2684],
         [ 1.1450,  0.1559],
         [ 1.3897, -0.5255],
         [ 0.0043,  0.2450],
         [ 0.0787, -0.5419],
         [ 0.5168, -0.5784],
         [ 1.0119,  1.9327],
         [-0.1872, -0.7539],
         [ 0.1311,  1.5708],
         [ 1.2899,  1.2745],
         [ 1.3265, -0.5562],
         [ 0.1075,  1.2083],
         [ 1.0381, -0.6970],
         [-0.1505, -0.4198],
         [-0.3288, -0.6862],
         [-0.8761,  0.3051],
         [ 0.1742, -0.0465],
         [ 0.7738,  0.5199],
         [-0.4224,  1.6996],
         [ 0.8065, -1.5457],
         [ 0.4880, -0.5391],
         [ 0.5614, -1.0774],
         [-0.9671,  0.5795]], requires_grad=True)]

### Indexing test

In [22]:
train_data[4]

tensor([ 0, 13,  1,  9, 20])

In [23]:
test_emb(train_data[4])

tensor([[-0.7809,  0.3950],
        [ 1.2899,  1.2745],
        [ 0.3863,  1.9058],
        [ 0.5168, -0.5784],
        [ 0.1742, -0.0465]], grad_fn=<IndexBackward0>)

In [24]:
test_emb(train_data[4, 0]), test_emb(train_data[4, 1]), test_emb(train_data[4, 2]), test_emb(train_data[4, 3]), test_emb(train_data[4, 4]) 

(tensor([-0.7809,  0.3950], grad_fn=<SelectBackward0>),
 tensor([1.2899, 1.2745], grad_fn=<SelectBackward0>),
 tensor([0.3863, 1.9058], grad_fn=<SelectBackward0>),
 tensor([ 0.5168, -0.5784], grad_fn=<SelectBackward0>),
 tensor([ 0.1742, -0.0465], grad_fn=<SelectBackward0>))

## Linear Layer

In [25]:
class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weights = torch.randn((fan_in, fan_out), requires_grad=True)
        self.bias = torch.zeros(fan_out, requires_grad=True) if bias else None

    def __call__(self, x):
        logits = x @ self.weights
        if self.bias != None:
            logits += self.bias

        return logits

    def parameters(self):
        params = [self.weights]
        if self.bias != None:
            params += [self.bias]

        return params

    def __repr__(self):
        return f"{self.__class__.__name__}: weights: {self.weights.shape} bias: {self.bias.shape}"

    def save_weights(self):
        return [p.detach().clone() for p in self.parameters()]

    def load_weights(self, weights):
        self.weights = weights[0].clone().requires_grad_()
        self.bias = weights[1].clone().requires_grad_() if len(weights) > 1 else None

In [26]:
l = Linear(3, 4)

In [27]:
l.weights, l.bias

(tensor([[-0.7246,  0.7924, -0.8122, -0.4553],
         [ 0.7624,  1.3441,  1.0642, -2.7180],
         [-0.1566, -0.8798,  0.6122, -0.1116]], requires_grad=True),
 tensor([0., 0., 0., 0.], requires_grad=True))

In [28]:
l.parameters()

[tensor([[-0.7246,  0.7924, -0.8122, -0.4553],
         [ 0.7624,  1.3441,  1.0642, -2.7180],
         [-0.1566, -0.8798,  0.6122, -0.1116]], requires_grad=True),
 tensor([0., 0., 0., 0.], requires_grad=True)]

## Tanh Layer

In [29]:
class Tanh:
    def __call__(self, x):
        return torch.tanh(x)

    def parameters(self):
        return []

    def __repr__(self):
        return f"{self.__class__.__name__}"

    def save_weights(self):
        return []

    def load_weights(self, weights):
        pass

In [30]:
x = torch.randn(1, 2)
x

tensor([[1.1563, 3.0990]])

In [31]:
t = Tanh()
t(x)

tensor([[0.8198, 0.9959]])

## Network

In [32]:
class Network:
    def __init__(self, alphabet_len, num_prefix_chars, num_embedding_dims, hidden_layers):
        self.alphabet_len = alphabet_len
        self.num_prefix_chars = num_prefix_chars
        self.num_embedding_dims = num_embedding_dims
        self.hidden_layers = hidden_layers

        self.embedding = Embedding(self.alphabet_len, self.num_embedding_dims)
        self.layers = [self.embedding]
        
        fan_in = self.num_prefix_chars * self.num_embedding_dims
        for h_size in hidden_layers:
            l = Linear(fan_in, h_size)
            torch.nn.init.kaiming_normal_(l.weights, nonlinearity="tanh")
            self.layers.append(l)
            self.layers.append(Tanh())
            fan_in = h_size

        last_layer = Linear(hidden_layers[-1], self.alphabet_len)
        with torch.no_grad():
            last_layer.weights *= 0.1
            
        self.layers.append(last_layer)

    def __call__(self, x):
        embedding = self.embedding(x)
        out = embedding.view(x.shape[0], self.num_prefix_chars * self.num_embedding_dims)
        for layer in self.layers[1:]:
            out = layer(out)

        return out

    def parameters(self):
        params = []
        for layer in self.layers:
            params += layer.parameters()


        return params

    def save_weights(self):
        return [l.save_weights() for l in self.layers]

    def load_weights(self, weights):
        assert len(self.layers) == len(weights)
        for l, w in zip(self.layers, weights):
            l.load_weights(w)            

    def __repr__(self):
        return f"{[l for l in self.layers]}"

In [33]:
n = Network(5, 5, 2, [1, 3, 2])
n.layers

[Embedding: shape(torch.Size([5, 2])),
 Linear: weights: torch.Size([10, 1]) bias: torch.Size([1]),
 Tanh,
 Linear: weights: torch.Size([1, 3]) bias: torch.Size([3]),
 Tanh,
 Linear: weights: torch.Size([3, 2]) bias: torch.Size([2]),
 Tanh,
 Linear: weights: torch.Size([2, 5]) bias: torch.Size([5])]

In [34]:
n.parameters()

[tensor([[-3.3107, -0.0675],
         [ 1.3056, -0.0578],
         [ 1.4279,  1.7692],
         [-1.2391,  0.0828],
         [-1.2585,  0.0739]], requires_grad=True),
 tensor([[ 1.5735],
         [-0.2495],
         [-2.1878],
         [-2.8965],
         [-0.7987],
         [-1.0658],
         [-0.0379],
         [-2.3156],
         [-2.7801],
         [-2.6962]], requires_grad=True),
 tensor([0.], requires_grad=True),
 tensor([[ 0.3937, -0.1599,  0.1952]], requires_grad=True),
 tensor([0., 0., 0.], requires_grad=True),
 tensor([[ 1.0519,  0.7226],
         [ 1.0141,  0.0603],
         [-0.7866,  1.4624]], requires_grad=True),
 tensor([0., 0.], requires_grad=True),
 tensor([[-0.2687, -0.1697,  0.0844,  0.1334, -0.1042],
         [-0.0635, -0.0299, -0.0593,  0.0941, -0.0950]], requires_grad=True),
 tensor([0., 0., 0., 0., 0.], requires_grad=True)]

## Training

In [35]:
def train(model, num_epochs, lr, batch_size):
    best_dev_loss = float('inf')
    best_epoch = -1
    best_config = ()

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        num_batches = 0
        perm = torch.randperm(len(train_data))
        
        for start in range(0, len(train_data), batch_size):
            num_batches += 1
            indexes = perm[start:start + batch_size]

            d = train_data[indexes]
            l = train_labels[indexes]
            
            logits = model(d)
            loss = F.cross_entropy(logits, l)
    
            for param in model.parameters():
                param.grad = None
    
            loss.backward()

            epoch_loss += loss.item()

            with torch.no_grad():
                for param in model.parameters():
                    param -= lr * param.grad

        with torch.no_grad():
            dev_loss = F.cross_entropy(model(dev_data), dev_labels)
            epoch_loss /= num_batches
            
            if epoch % 10 == 0:
                print(f"{epoch=}: {epoch_loss=}, {dev_loss=}")

            if dev_loss < best_dev_loss:
                best_dev_loss = dev_loss
                best_epoch = epoch
                best_config = {
                    "best_loss": best_dev_loss.item(), 
                    "best_epoch": best_epoch,
                    "num_embedding_dims": model.num_embedding_dims,
                    "hidden_layer_size": model.hidden_layers[0],
                    "weights": model.save_weights()
                }

    return best_config


In [36]:
hidden_layer_sizes = [100, 200, 300]
lr = 0.1
dim_sizes = [5, 10, 15, 20]
num_epochs = 150
batch_size = 64

best_dev_loss = float('inf')
best_config = None
for hidden_layer_size in hidden_layer_sizes:
    for num_embedding_dims in dim_sizes:
        print(f"==== Starting training with {num_embedding_dims=} dims and {hidden_layer_size=} ====")
        model = Network(alphabet_len, num_prefix_chars, num_embedding_dims, [hidden_layer_size]) 
        print(f"{model=}")

        config = train(model, num_epochs, lr, batch_size)
        if config["best_loss"] < best_dev_loss:
            best_dev_loss = config["best_loss"]
            best_config = config
            print(f"Best config: {best_config["num_embedding_dims"]=}, {best_config["hidden_layer_size"]=}, {best_config["best_loss"]=}")

print(f"Best config: {best_config["num_embedding_dims"]=}, {best_config["hidden_layer_size"]=}, {best_config["best_loss"]=}")

==== Starting training with num_embedding_dims=5 dims and hidden_layer_size=100 ====
model=[Embedding: shape(torch.Size([27, 5])), Linear: weights: torch.Size([25, 100]) bias: torch.Size([100]), Tanh, Linear: weights: torch.Size([100, 27]) bias: torch.Size([27])]
epoch=0: epoch_loss=2.42085401960023, dev_loss=tensor(2.3249)
epoch=10: epoch_loss=2.139274312504046, dev_loss=tensor(2.1729)
epoch=20: epoch_loss=2.1141364061385026, dev_loss=tensor(2.1406)
epoch=30: epoch_loss=2.103014099518486, dev_loss=tensor(2.1325)
epoch=40: epoch_loss=2.095959965114801, dev_loss=tensor(2.1296)
epoch=50: epoch_loss=2.0916032679078453, dev_loss=tensor(2.1347)
epoch=60: epoch_loss=2.088375510980438, dev_loss=tensor(2.1370)
epoch=70: epoch_loss=2.086626801520889, dev_loss=tensor(2.1430)
epoch=80: epoch_loss=2.08492327463999, dev_loss=tensor(2.1393)
epoch=90: epoch_loss=2.0833900211485425, dev_loss=tensor(2.1386)
epoch=100: epoch_loss=2.0825127357769246, dev_loss=tensor(2.1352)
epoch=110: epoch_loss=2.080855

In [37]:
for lr in [0.01, 0.001]:
    model = Network(alphabet_len, num_prefix_chars, best_config["num_embedding_dims"], [best_config["hidden_layer_size"]])
    model.load_weights(best_config["weights"])
    
    config = train(model, 50, lr, batch_size)
    if config["best_loss"] < best_dev_loss:
        best_dev_loss = config["best_loss"]
        best_config = config
        print(f"Best config: {best_config["num_embedding_dims"]=}, {best_config["hidden_layer_size"]=}, {best_config["best_loss"]=}")

print(f"Best config: {best_config["num_embedding_dims"]=}, {best_config["hidden_layer_size"]=}, {best_config["best_loss"]=}")
    

epoch=0: epoch_loss=1.8835989426663364, dev_loss=tensor(2.0160)
epoch=10: epoch_loss=1.8682526873940126, dev_loss=tensor(2.0183)
epoch=20: epoch_loss=1.8630899714821474, dev_loss=tensor(2.0228)
epoch=30: epoch_loss=1.8588202467971582, dev_loss=tensor(2.0239)
epoch=40: epoch_loss=1.8551190616958226, dev_loss=tensor(2.0240)
Best config: best_config["num_embedding_dims"]=20, best_config["hidden_layer_size"]=200, best_config["best_loss"]=2.013742208480835
epoch=0: epoch_loss=1.8677335337813132, dev_loss=tensor(2.0118)
epoch=10: epoch_loss=1.8651862648736799, dev_loss=tensor(2.0115)
epoch=20: epoch_loss=1.864027001029021, dev_loss=tensor(2.0123)
epoch=30: epoch_loss=1.8630298954777365, dev_loss=tensor(2.0122)
epoch=40: epoch_loss=1.8621293645873482, dev_loss=tensor(2.0130)
Best config: best_config["num_embedding_dims"]=20, best_config["hidden_layer_size"]=200, best_config["best_loss"]=2.0114221572875977
Best config: best_config["num_embedding_dims"]=20, best_config["hidden_layer_size"]=200,

In [38]:
best_model = Network(alphabet_len, num_prefix_chars, best_config["num_embedding_dims"], [best_config["hidden_layer_size"]])
best_model.load_weights(best_config["weights"])

In [39]:
def predict_next_char_idx(model, prefix):
    with torch.no_grad():
        logits = model(prefix)
        index = torch.multinomial(F.softmax(logits, dim=1), num_samples=1, replacement=True)

        return index[0].item()

In [40]:
def generate_name(model):
    name = ""
    prefix = [char_to_idx['.']] * num_prefix_chars
    
    while True:
        next_idx = predict_next_char_idx(model, torch.tensor(prefix).unsqueeze(0))
        
        if next_idx == 0:
            break
            
        name += alphabet[next_idx]
        prefix = prefix[1:] + [next_idx]
    
    return name    

In [41]:
generate_name(best_model)

'marli'

In [42]:
def generate_names(model, num_names):
    names = []
    for _ in range(num_names):
        names.append(generate_name(model))


    return names

In [43]:
gen_names = generate_names(best_model, 10)
gen_names

['mayah',
 'aalee',
 'breyah',
 'eevan',
 'aquaria',
 'lindio',
 'adyline',
 'khairyn',
 'lamya',
 'damiyla']

In [44]:
in_dataset = 0
for n in gen_names:
    if n in training_set:
        in_dataset += 1

in_dataset

3

In [45]:
with torch.no_grad():
    test_loss = F.cross_entropy(best_model(test_data), test_labels)
    print(f"Test loss: {test_loss.item():4f}")

Test loss: 2.030257
